In [12]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
#from sklearn.cluster import KMeans
from PIL import Image
import os
import pandas as pd
import numpy as np
#from sklearn.cluster import KMeans
import shutil
from collections import Counter
import json
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [15]:
# load resNet 50 model
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.0406], std=[0.229, 0.224, 0.225])
])

c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [16]:
def extract_features(path):
    img = Image.open(path).convert('RGB')
    t = transform(img).unsqueeze(0)
    with torch.no_grad():
        return model(t).flatten().numpy()


# paths
samples = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\k-means_training\fabrics'
fabric_csv = "fabric_analysis_results.csv"

print("learning fabrics...")
X_train = []
y_train = []

for fabric_type in os.listdir(samples):
    folder_path = os.path.join(samples, fabric_type)
    
    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):
            if img_name.lower().endswith(('.png', 'jpg', '.jpeg')):
                all_features = extract_features(os.path.join(folder_path, img_name))
                X_train.append(all_features)
                y_train.append(fabric_type)

learning fabrics...


In [17]:
# SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

clf = SVC(kernel='linear', class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)
print(f'DONE TRAINING {len(X_train)} images')

#predicitions pleaseeee
print('testing...')
results = []

for root, dirs, files in os.walk(runway):
    for img_name in files:
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_paths = os.path.join(root, img_name)

            try:
                curr_feature = extract_features(img_paths)
                curr_scaled = scaler.transform([curr_features])

                #pred
                pred = clf.predict(curr_scaled)[0]
                brand_name = os.path.basename(root)

                results.append({
                    "brand": brand_name,
                    "image": img_name,
                    "fabric": pred})

            except Exception as e:
                print(f"Skipping {img_name} due to {e}")

DONE TRAINING 72 images
testing...


In [18]:
# analyzing the dataset
runway = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\runway'
results = []

print("classifying runway fabrics using samples...")
for root, dirs, files in os.walk(runway):
    for img_name in files:
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_paths = os.path.join(root, img_name)

            try:
                curr_features = extract_features(img_paths)

                #best match to samples
                best_match = None
                min_dist = float('inf')

                for name, ref_features in avg_features.items():
                    dist = np.linalg.norm(curr_features - ref_features)
                    if dist < min_dist:
                        min_dist = dist
                        best_match = name
                
                brand_name = os.path.basename(root)
                results.append({
                    "brand": brand_name,
                    "image": img_name,
                    "fabric": best_match
    })
            except Exception as e:
                print(f"Skipping {img_name} due to {e}")

classifying runway fabrics using samples...


KeyboardInterrupt: 

In [19]:
# percentage
#extract fabric names
fabric_name = [item['fabric'] for item in results]

counts = Counter(fabric_name)
total = len(fabric_name)
print("\n--- FINAL FABRIC TRENDS ---")
for fabric, count in counts.items():
    perc = (count/total) * 100
    print(f"{fabric}: {perc:.1f}%")


--- FINAL FABRIC TRENDS ---
silk: 100.0%


In [10]:
import shutil
import os

# 1. Define where you want the sorted folders to go
output_visual_dir = r"C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\outputs\fabric_check"

print("Organizing images into fabric folders for visual verification...")

for item in results:
    fabric_name = item['fabric']
    img_name = item['image']
    brand_name = item['brand']
    
    # We need the original path to the image to copy it
    # This assumes your runway images are in brand subfolders
    original_path = os.path.join(runway, brand_name, img_name)
    
    # Create the destination folder (e.g., .../fabric_check/silk)
    target_folder = os.path.join(output_visual_dir, fabric_type)
    os.makedirs(target_folder, exist_ok=True)
    
    # Copy the file
    try:
        shutil.copy(original_path, os.path.join(target_folder, img_name))
    except Exception as e:
        print(f"Could not copy {img_name}: {e}")

print(f"Done! Go to: {output_visual_dir}")

Organizing images into fabric folders for visual verification...
Done! Go to: C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\outputs\fabric_check


JSON AND CSV EXTRACT

In [ ]:
df = pd.DataFrame(results)
df.to_csv(fabric_csv, index=False)

summary = dict(Counter(df['fabric']))
for_frontend = {
    "summary": summary,
    "percentages": {dist: round((v/len(df))*100,2) for dist, v in summary.items()},
    "details": results
}
json_output = "fabric_results.json"
with open(json_output, 'w') as f:
    json.dump(for_frontend, f, indent=4)

print("fabric analysis done and saved to csv and json paths")